<a href="https://colab.research.google.com/github/satvik0777/Brazilian-E-Commerce-Public-Dataset-by-Olist/blob/main/Day%201/Olist_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛒 Olist E-Commerce — Day 1: Data Loading & Cleaning

**Goal:** Load all 9 Olist datasets via KaggleHub, inspect their structure, check for null values, and perform initial cleaning on the datasets that need it.

## ⚙️ Setup — Install Dependencies

In [ ]:
!pip install -q kagglehub[pandas-datasets] pandas

## 📦 Import Libraries

In [ ]:
import pandas as pd

## 📂 Load Datasets

All 9 CSV files from the [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) are loaded below.

| Variable | File |
|----------|------|
| `cd_df` | olist_customers_dataset |
| `ogd_df` | olist_geolocation_dataset |
| `ooid_df` | olist_order_items_dataset |
| `oopd_df` | olist_order_payments_dataset |
| `oord_df` | olist_order_reviews_dataset |
| `ood_df` | olist_orders_dataset |
| `opd_df` | olist_products_dataset |
| `osd_df` | olist_sellers_dataset |
| `pcnt_df` | product_category_name_translation |

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

cd_df   = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_customers_dataset.csv')
ogd_df  = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_geolocation_dataset.csv')
ooid_df = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_order_items_dataset.csv')
oopd_df = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_order_payments_dataset.csv')
oord_df = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_order_reviews_dataset.csv')
ood_df  = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_orders_dataset.csv')
opd_df  = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_products_dataset.csv')
osd_df  = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'olist_sellers_dataset.csv')
pcnt_df = kagglehub.load_dataset('olistbr/brazilian-ecommerce', 'product_category_name_translation.csv')

print('All datasets loaded successfully!')

## 👀 Quick Peek — Customers Dataset

In [ ]:
cd_df.head()

## 🗂️ Dataset Info — Shape & Dtypes

Inspecting structure (columns, non-null counts, dtypes) for all 9 datasets.

In [ ]:
for name, df in [
    ('Customers', cd_df), ('Geolocation', ogd_df), ('Order Items', ooid_df),
    ('Order Payments', oopd_df), ('Order Reviews', oord_df), ('Orders', ood_df),
    ('Products', opd_df), ('Sellers', osd_df), ('Category Translation', pcnt_df)
]:
    print(f'\n{'='*40}')
    print(f'  {name}  |  Shape: {df.shape}')
    print('='*40)
    df.info()


## 🔍 Null Value Check — All Datasets

Checking missing values across every table to decide what needs cleaning.

In [ ]:
for name, df in [
    ('Customers', cd_df), ('Geolocation', ogd_df), ('Order Items', ooid_df),
    ('Order Payments', oopd_df), ('Order Reviews', oord_df), ('Orders', ood_df),
    ('Products', opd_df), ('Sellers', osd_df), ('Category Translation', pcnt_df)
]:
    nulls = df.isnull().sum()
    if nulls.any():
        print(f'\n[{name}]')
        print(nulls[nulls > 0])
    else:
        print(f'\n[{name}] — No nulls ✅')


## 🧹 Cleaning — Order Reviews Dataset

`review_comment_title` and `review_comment_message` have many nulls — customers who left a star rating but no written comment. Filling with `'No comment'`.

In [ ]:
oord_df.head()

In [ ]:
oord_df['review_comment_title']   = oord_df['review_comment_title'].fillna('No comment')
oord_df['review_comment_message'] = oord_df['review_comment_message'].fillna('No comment')
print('Nulls remaining in Order Reviews:')
print(oord_df.isnull().sum())

## 🧹 Cleaning — Orders Dataset

Converting all timestamp columns to `datetime`. Null delivery dates are expected for non-delivered orders — we handle them by creating a `completion_status` column instead of dropping rows.

In [ ]:
ood_df.head()

In [ ]:
timestamp_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in timestamp_cols:
    ood_df[col] = pd.to_datetime(ood_df[col])

print('Nulls after datetime conversion:')
print(ood_df.isnull().sum())

In [ ]:
# Orders with null delivery date — check their statuses
null_delivery = ood_df[ood_df['order_delivered_customer_date'].isna()]['order_status'].value_counts()
print('Order statuses where delivery date is null:')
print(null_delivery)

In [ ]:
# Derive a high-level completion_status column
def order_completion_status(row):
    if row['order_status'] == 'delivered':
        return 'Delivered'
    elif row['order_status'] in ['shipped', 'processing', 'approved', 'invoiced']:
        return 'Stuck / In Progress'
    elif row['order_status'] in ['cancelled', 'unavailable']:
        return 'Cancelled'
    else:
        return 'Other'

ood_df['completion_status'] = ood_df.apply(order_completion_status, axis=1)
print('Order completion status distribution:')
print(ood_df['completion_status'].value_counts())

## 🧹 Cleaning — Products Dataset

- Rows with no `product_category_name` are dropped (cannot be categorised meaningfully).
- Missing physical dimensions (`weight`, `length`, `height`, `width`) are filled with the **median** of each column.

In [ ]:
opd_df.head()

In [ ]:
opd_df.dropna(subset=['product_category_name'], inplace=True)

dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dim_cols:
    opd_df[col] = opd_df[col].fillna(opd_df[col].median())

print('Nulls remaining in Products:')
print(opd_df.isnull().sum())

## ✅ Day 1 Summary

| Dataset | Action Taken |
|---------|-------------|
| Customers | No nulls — ready to use ✅ |
| Geolocation | No nulls — ready to use ✅ |
| Order Items | No nulls — ready to use ✅ |
| Order Payments | No nulls — ready to use ✅ |
| **Order Reviews** | Filled comment nulls with `'No comment'` |
| **Orders** | Converted timestamps to datetime; added `completion_status` column |
| **Products** | Dropped rows with no category; filled dimension nulls with median |
| Sellers | No nulls — ready to use ✅ |
| Category Translation | No nulls — ready to use ✅ |

**Next up (Day 2):** Exploratory Data Analysis — sales trends, top categories, order volume over time.